In [1]:
%pip install pandas matplotlib openpyxl jupyter nbconvert nbformat

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.9 MB 7.0 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 8.7 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 9.7 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/9.9 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 10.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --------- ------------------------------ 2.1/9.3 MB 10.7 MB/s eta 0:00:01
   ------------------- -------------------- 4.5/9.3 MB 11.2 MB/s eta 0:00:01
   ---------------------------- ----------- 6.6/9.3 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 11.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -----------------------------

# deka — Reproducible pipeline: raw data → metadata → dashboard

This single notebook is the **entire data-preparation pipeline** for the Bank XYZ
branch customer-experience dashboard. Run it top-to-bottom and it regenerates the
three metadata artifacts the app consumes:

| Output | Role |
|---|---|
| `metadata/metadata.csv` | the *semantic ("friend") layer* — every variable tagged with Touchpoint / Construct / Dashboard Role (documented intermediate) |
| `metadata/metadata_dashboard.csv` | **the file `dashboard.py` reads** — 479 rows, one per dashboard variable, with `section / touchpoint / subgroup / role / include` |
| `metadata/metadata_dashboard.xlsx` | hand-editable control surface for the same 479 rows (toggle `include`, rename `subgroup`/`label`) |

After running this notebook, `streamlit run dashboard.py` works with **nothing else** —
the reproducible end state is just **raw data + this notebook + `dashboard.py`**.

**How to reproduce**
```bash
cd ~/deka                     # WSL-native filesystem, NOT /mnt/c
uvx --with pandas,openpyxl,plotly,matplotlib,nbformat,nbconvert \
    jupyter nbconvert --to notebook --execute --inplace pipeline.ipynb
streamlit run dashboard.py
```

> **Scope note.** This notebook *reorganizes and narrates* the original
> `eda/EDA.ipynb` + `preprocessing/buat_metadata.py` chain — it does **not** redesign
> any classification, scoring, or include logic. Outputs are reproduced byte-for-byte
> so `dashboard.py` runs unchanged. The EDA section adds exploration/visuals to explain
> *why* each preprocessing choice is made, but never alters the choices themselves.

## 1 · Setup & load the raw survey

The raw file is **semicolon-delimited** with a two-line header:

- **row 0** = full Indonesian question text (used as `question` / chart labels)
- **row 1** = short variable codes (e.g. `E1A`, `T_AT1_1`) — the real column names

So data is read with `header=1` (codes become columns) while row 0 is read separately
as the question dictionary. 1,730 respondents × 632 variables.

In [2]:
import os, re
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless-safe, deterministic
import matplotlib.pyplot as plt

# Paths resolve off the project root (run this notebook from ~/deka).
ROOT = Path.cwd()
DATA = ROOT / "data" / "Deka_project_dataset_BankXYZ.csv"
# Output dir is overridable for scratch/testing; defaults to the real metadata/ folder.
OUT_DIR = Path(os.environ.get("DEKA_METADATA_OUT", ROOT / "metadata"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT     :", ROOT)
print("DATA     :", DATA, "| exists:", DATA.exists())
print("OUT_DIR  :", OUT_DIR)

ROOT     : c:\Users\Muhammad Hafiz F\Documents\ali\deka done
DATA     : c:\Users\Muhammad Hafiz F\Documents\ali\deka done\data\Deka_project_dataset_BankXYZ.csv | exists: True
OUT_DIR  : c:\Users\Muhammad Hafiz F\Documents\ali\deka done\metadata


In [3]:
# Full respondent table: codes as columns (header=1), values as raw strings.
df = pd.read_csv(DATA, sep=";", header=1, low_memory=False, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
print(f"shape: {df.shape[0]:,} respondents x {df.shape[1]} variables")
df.iloc[:3, :6]

shape: 1,730 respondents x 632 variables


,SERIAL,PROV,KABKOTA,CABANG,S1,S2_1
0,197513878,DKI Jakarta,Jakarta Selatan,Jakarta Selatan 1,Pria,26
1,197518398,DKI Jakarta,Jakarta Timur,Jakarta Timur 1,Pria,43
2,197544291,DKI Jakarta,Jakarta Timur,Jakarta Timur 1,Pria,30


## 2 · EDA — structure, messiness, and the *why* behind each preprocessing choice

Before building metadata we need to understand three things about the raw data, because
each one drives a downstream decision:

1. **Two-row header** → we need a variable→question dictionary.
2. **Ratings are stored as coded strings** (`"6  SANGAT PUAS"`, `"99 TIDAK RELEVAN"`) →
   we need a robust numeric extractor that drops N/A codes.
3. **~42% of cells are empty *by design*** (conditional routing, a competitor-only subset,
   and `99/999` "not applicable" codes) → we must **never impute**; missingness is signal.

In [4]:
# --- the variable -> question dictionary (built from the two header rows) ---
headers_df = pd.read_csv(DATA, sep=";", header=None, nrows=2)
long_descriptions = headers_df.iloc[0].tolist()   # row 0: question text
short_names       = headers_df.iloc[1].tolist()   # row 1: variable codes
data_dict = dict(zip(short_names, long_descriptions))
print(f"{len(data_dict)} variables in the data dictionary")
pd.DataFrame({"variable": list(data_dict)[:8],
              "question": [data_dict[v] for v in list(data_dict)[:8]]})

632 variables in the data dictionary


,variable,question
0,SERIAL,Serial ID
1,PROV,Provinsi
2,KABKOTA,Kab / Kota
3,CABANG,NAMA KANTOR CABANG
4,S1,Jenis kelamin
5,S2_1,Berapa tahun usia Anda? __ tahun
6,S2_2,Range Usia
7,S3,Apakah Bapak/Ibu merupakan nasabah (pemilik re...


### 2.1 · Ratings are coded strings — `clean_ratings`

Satisfaction items look like `"6  SANGAT PUAS"` / `"1 SANGAT TIDAK PUAS"`. "Not applicable"
is coded `99` / `999`, and routed-out questions are blank. The extractor pulls the leading
integer **only when it precedes ALL-CAPS label text** (a real rating), and maps `99*`, blanks,
and free text to `NaN`. This is the same idea the dashboard's `to_score` uses
(`to_score` additionally caps at the scale max, e.g. 6).

In [5]:
def clean_ratings(val):
    if pd.isna(val) or str(val).strip() == "":          # blank / routed-out
        return float("nan")
    val_str = str(val).strip()
    if val_str.startswith("99"):                         # 99 / 999 = Not Applicable
        return float("nan")
    m = re.match(r"^(\d+)\s+[A-Z]", val_str)             # "6  SANGAT PUAS" -> 6
    if m:
        return float(m.group(1))
    if val_str.isdigit():                                # already a bare number
        return float(val_str)
    return val                                           # city names etc. left as-is

demo = pd.Series(["6  SANGAT PUAS", "1 SANGAT TIDAK PUAS", "99 TIDAK RELEVAN",
                  "", "JAKARTA", "5"])
pd.DataFrame({"raw": demo, "clean_ratings": demo.map(clean_ratings)})

,raw,clean_ratings
0,6 SANGAT PUAS,6.0
1,1 SANGAT TIDAK PUAS,1.0
2,99 TIDAK RELEVAN,NaN
3,,NaN
4,JAKARTA,JAKARTA
5,5,5.0


### 2.2 · ~42% missing — by design, never imputed

Overall emptiness is high, but it is **structural**, not data loss:

- **Conditional routing** — a respondent who never used the ATM never sees ATM attribute
  items, so those cells are blank for them.
- **Competitor subset** — `*B` / competitor items (e.g. `E1B`, `G1C`) are only asked of the
  relevant sub-sample.
- **`99` / `999` N/A codes** — explicit "not applicable", mapped to `NaN` above.

Because the blanks carry meaning (this respondent wasn't in scope), we compute every metric
on the **answered** rows only and never fill. This is also why, downstream, a variable is
marked `include=1` **only if it exists in the data *and* has at least one answered row**.

In [6]:
overall_missing = df.isna().mean().mean() * 100
print(f"Overall empty cells: {overall_missing:0.1f}%  (blank routing + competitor subset + 99/999)")

# Missingness for the headline KPI columns (illustrative; competitor items are sparser).
kpi_cols = [c for c in ["E1A","E1B","F1A","F1B","G1A","G1C"] if c in df.columns]
miss = (df[kpi_cols].isna().mean() * 100).sort_values()
ax = miss.plot.bar(color="#0F4C81", figsize=(7,3))
ax.set_ylabel("% missing"); ax.set_title("Missingness by KPI column (XYZ vs competitor)")
plt.tight_layout(); plt.show()

Overall empty cells: 0.0%  (blank routing + competitor subset + 99/999)


C:\Users\Muhammad Hafiz F\AppData\Local\Temp\ipykernel_32528\947960069.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


> **Note (classification, unchanged).** The `T_C1A*/T_C1B*` brand-attribute items are
> arguably *Brand Image* by construct. The committed friend layer historically tags them
> `Construct = Other` (only their **Touchpoint = Brand Image** is set), and the dashboard
> selects Brand-Image attributes via the `T_C1B` prefix — so the `Other` tag is harmless.
> We **flag** this here but **do not change it**, to keep the output byte-identical.

## 3 · Build the semantic ("friend") metadata layer

Every variable is tagged by three deterministic rule functions:

- **`assign_touchpoint`** — prefix → touchpoint (`T_AT*`→ATM, `T_TL*`→Teller, `T_CS*`→Customer
  Service, `T_CA*`→Customer Advisor, `T_KC*`→Branch Facilities, `T_SC*`→Security, `T_SL*`→Service
  Electronics, `T_C1*`→Brand Image, `E/F/G/T_H/T_I/T_J*`→Overall).
- **`assign_construct`** — what the item *measures* (CSAT, NPS, Loyalty, Emotion, Attribute
  Satisfaction, Overall Touchpoint Satisfaction, Demographic, Usage, Open Ended, …).
- **`assign_dashboard_role`** — construct → role (`KPI`, `Drilldown`, `Filter`, `Benchmark`, …),
  which the next stage maps onto dashboard pages.

The result is written to `metadata/metadata.csv` (CRLF, no BOM, to match the committed file).

In [7]:
def assign_touchpoint(var):
    if var.startswith("T_AT"): return "ATM"
    elif var.startswith("T_TL"): return "Teller"
    elif var.startswith("T_CS"): return "Customer Service"
    elif var.startswith("T_CA"): return "Customer Advisor"
    elif var.startswith("T_KC"): return "Branch Facilities"
    elif var.startswith("T_SC"): return "Security"
    elif var.startswith("T_SL"): return "Service Electronics"
    elif var.startswith(("E", "F", "G", "T_H", "T_I", "T_J")): return "Overall"
    elif var.startswith("T_C1"): return "Brand Image"
    else: return None

def assign_construct(var, question):
    q = str(question).lower()
    if "alasan" in q: return "Open Ended"
    if re.match(r"^(S\d|P\d)", var): return "Demographic"
    elif re.match(r"^(A|B|D|AT1|TL1|CS1)", var): return "Usage"
    elif var.startswith("G1"): return "NPS"
    elif var.startswith("E1"): return "CSAT"
    elif var.startswith(("F1", "T_H")): return "Loyalty"
    elif var.startswith("T_I"): return "Emotion"
    elif var.startswith("T_J"): return "Digitalization"
    elif "penilaian keseluruhan" in q: return "Overall Touchpoint Satisfaction"
    elif var.startswith(("T_AT","T_TL","T_CS","T_CA","T_KC","T_SC","T_SL")):
        return "Attribute Satisfaction"
    # NOTE: the committed friend layer intentionally has NO `T_C1A/B -> Brand Image`
    # branch here (see the EDA note above) -> those items fall through to "Other".
    return "Other"

def assign_dashboard_role(construct):
    if construct == "Demographic": return "Filter"
    elif construct in ["CSAT","NPS","Loyalty","Overall Touchpoint Satisfaction"]: return "KPI"
    elif construct == "Attribute Satisfaction": return "Drilldown"
    elif construct == "Brand Image": return "Benchmark"
    elif construct == "Open Ended": return "Text Analysis"
    return "Other"

metadata = pd.DataFrame({"variable": list(data_dict.keys()),
                         "question": list(data_dict.values())})
metadata["Touchpoint"] = metadata["variable"].apply(assign_touchpoint)
metadata["Construct"]  = metadata.apply(
    lambda x: assign_construct(x["variable"], x["question"]), axis=1)
metadata["Dashboard Role"] = metadata["Construct"].apply(assign_dashboard_role)

# CRLF + no BOM == byte-identical to the committed Windows-generated file.
metadata.to_csv(OUT_DIR / "metadata.csv", index=False,
                encoding="utf-8", lineterminator="\r\n")
print(f"wrote {OUT_DIR/'metadata.csv'}  ({len(metadata)} rows)")
metadata["Dashboard Role"].value_counts()

wrote c:\Users\Muhammad Hafiz F\Documents\ali\deka done\metadata\metadata.csv  (632 rows)


Dashboard Role
Drilldown        400
Other            155
KPI               54
Filter            14
Text Analysis      9
Name: count, dtype: int64

In [8]:
# Sanity peek: how the friend layer distributes across touchpoints & constructs.
display(metadata["Touchpoint"].value_counts(dropna=False).to_frame("n_variables"))
display(metadata["Construct"].value_counts().to_frame("n_variables"))

,n_variables
Touchpoint,
Branch Facilities,113
Overall,87
Brand Image,72
Customer Service,71
Teller,59
ATM,56
NaN,55
Security,47
Customer Advisor,39


,n_variables
Construct,
Attribute Satisfaction,400
Other,95
Loyalty,32
Emotion,32
Usage,18
Overall Touchpoint Satisfaction,18
Demographic,14
Digitalization,10
Open Ended,9


## 4 · Build the dashboard metadata (`metadata_dashboard.{csv,xlsx}`)

This stage turns the friend layer into the exact 479-row table `dashboard.py` consumes.
Logic is ported verbatim from `preprocessing/buat_metadata.py`:

- **`SECTION_OF_TP`** maps each touchpoint onto one of the 5 dashboard pages
  (Customer Service / Teller / Security / Customer Advisor / Service Electronics all collapse
  into **Service Experience**).
- **`classify_subgroup`** assigns a sub-category per attribute from keyword rules on the
  question text (word-boundary match so `aman` doesn't fire on `nyaman`); first matching rule
  wins, hence most-specific rules sit on top.
- **`include = exists_in_data AND answered_rows > 0`.** Competitor (`*B` / `G1C`) attribute
  rows are kept in the metadata for completeness but ship `include=0` — the dashboard only
  uses competitor *KPIs* (`E1B`,`F1B`,`G1C`) for the XYZ-vs-competitor comparison.

> ⚠️ **Load-bearing metadata contract.** `dashboard.py` hardcodes the exact column names
> (`variable, question, label, bank, section, touchpoint, subgroup, role, scale_max, include,
> n_terisi, ada_di_data`) and the exact values it filters on (`role ∈ {Atribut, Overall,
> Filter*}`, `bank=="XYZ"`, page names like `"Brand Image"`, touchpoints like `"Customer
> Service"`). Changing any of these makes pages silently render empty. Do not alter them.

In [9]:
# --- KPI, filter and section/ordering constants (verbatim) ---
KPI_VARS = [
    ("E1A", "CSAT — Kepuasan keseluruhan", "XYZ", 6),
    ("E1B", "CSAT — Kepuasan keseluruhan", "Kompetitor", 6),
    ("F1A", "Loyalty — Kesediaan terus menggunakan", "XYZ", 6),
    ("F1B", "Loyalty — Kesediaan terus menggunakan", "Kompetitor", 6),
    ("G1A", "NPS — Kemungkinan merekomendasikan (0–10)", "XYZ", 10),
    ("G1C", "NPS — Kemungkinan merekomendasikan (0–10)", "Kompetitor", 10),
]
FILTER_VARS = [
    ("PROV", "Provinsi", "Utama"), ("KABKOTA", "Kabupaten/Kota", "Utama"),
    ("CABANG", "Cabang", "Utama"), ("S2_2", "Kelompok Usia", "Utama"),
    ("S4", "Lama Menjadi Nasabah", "Utama"), ("S1", "Gender", "Tambahan"),
    ("S7", "Frekuensi Transaksi", "Tambahan"),
]
PROFIL_VARS = ["S1", "S2_2", "S4"]
SECTION_OF_TP = {
    "Brand Image": "Brand Image", "Branch Facilities": "Branch Facilities",
    "Customer Service": "Service Experience", "Teller": "Service Experience",
    "Security": "Service Experience", "Customer Advisor": "Service Experience",
    "Service Electronics": "Service Experience", "ATM": "ATM Experience",
}
SECTION_ORDER = ["Ringkasan", "Brand Image", "Branch Facilities",
                 "Service Experience", "ATM Experience", "Filter & Profil"]
TP_ORDER = list(SECTION_OF_TP.keys())

In [10]:
# --- sub-category keyword rules (most-specific first); verbatim from buat_metadata.py ---
RULES_BF = [
    ("Toilet", ["toilet"]), ("Parkir", ["parkir"]),
    ("Ruang Tunggu & Antrian", ["tunggu","menunggu","duduk","antri","antrian","mengantri"]),
    ("Banking Hall & Kenyamanan", ["banking hall","interior","bersih","kebersihan","suhu","sejuk","ac","pencahayaan","nyaman","kenyamanan","ruang"]),
    ("Lokasi & Akses", ["lokasi","akses","gedung","strategis","papan nama","tampak","dijangkau","jangkau"]),
    ("Sarana Pendukung", ["formulir","slip","brosur","alat tulis","writing","mesin","atm di cabang"]),
]
RULES_BI = [
    ("Kepercayaan & Reputasi", ["aman","keamanan","percaya","kepercayaan","terpercaya","reputasi","kinerja","dihargai","mengontrol"]),
    ("Citra & Kebanggaan", ["bangga","kebanggaan","prestis","bergengsi","terkenal","terkemuka","digunakan banyak","negara","swasta terbesar"]),
    ("Kemudahan & Teknologi", ["mudah","kemudahan","mempermudah","transaksi","channel","teknologi","online","fitur","e-channel","atm","kantor cabang","call center"]),
    ("Produk & Manfaat", ["untung","menguntungkan","keuntungan","investasi","bisnis","diskon","produk","promo","lengkap","layanan"]),
]
RULES_PETUGAS = [
    ("Penampilan", ["penampilan","rapi","seragam","menarik","berwibawa"]),
    ("Sikap & Keramahan", ["ramah","keramahan","sopan","senyum","salam","sapa","menyapa","sikap","membantu","peduli","perhatian","membukakan"]),
    ("Kecepatan & Ketanggapan", ["cepat","kecepatan","waktu","antri","tanggap","sigap","segera"]),
    ("Kompetensi & Solusi", ["mampu","kemampuan","pengetahuan","menjelaskan","informasi","solusi","akurat","teliti","ketelitian","paham","memahami","mengerti","jawab","kompeten","keluhan","masalah","kebutuhan","menghitung","memproses"]),
]
RULES_SECURITY = [("Keamanan & Kesigapan", ["aman","keamanan","menjaga","jaga","sigap","mengarahkan"])] + RULES_PETUGAS
RULES_ELEKTRONIK = [
    ("Mesin & Perangkat", ["mesin","perangkat","edc","tablet","layar","e-form"]),
    ("Kemudahan Penggunaan", ["mudah","kemudahan","jelas","petunjuk","informasi"]),
    ("Keandalan", ["berfungsi","gangguan","rusak","kerusakan","cepat"]),
]
RULES_ATM = [
    ("Keamanan", ["aman","keamanan","cctv","penjaga","dijaga"]),
    ("Keandalan Mesin", ["gangguan","rusak","kerusakan","offline","kartu","macet","berfungsi","kehabisan","tertelan","error"]),
    ("Fitur & Transaksi", ["fitur","menu","pecahan","setor","tarik","transfer","pembayaran","lengkap","transaksi","uang"]),
    ("Ketersediaan & Lokasi", ["lokasi","jumlah","tersedia","ketersediaan","mudah ditemukan","dekat","banyak","mencukupi"]),
    ("Kenyamanan & Kebersihan", ["bersih","kebersihan","nyaman","kenyamanan","ruang","suhu","terang","pencahayaan"]),
]
RULES_BY_TP = {
    "Brand Image": (RULES_BI, "Citra Umum"),
    "Branch Facilities": (RULES_BF, "Fasilitas Lain"),
    "Customer Service": (RULES_PETUGAS, "Pelayanan Umum"),
    "Teller": (RULES_PETUGAS, "Pelayanan Umum"),
    "Security": (RULES_SECURITY, "Pelayanan Umum"),
    "Customer Advisor": (RULES_PETUGAS, "Pelayanan Umum"),
    "Service Electronics": (RULES_ELEKTRONIK, "Sarana Lain"),
    "ATM": (RULES_ATM, "ATM Umum"),
}

def classify_subgroup(touchpoint, question):
    rules, default = RULES_BY_TP.get(touchpoint, ([], "Lainnya"))
    q = " " + str(question).lower() + " "
    for name, keywords in rules:
        # word-boundary match so 'aman' does not fire inside 'nyaman'
        if any(re.search(r"\b" + re.escape(k), q) for k in keywords):
            return name
    return default

In [11]:
# --- scoring / labelling helpers + friend-frame normalization (verbatim logic) ---
def to_score(series, max_valid=6):
    s = series.astype(str).str.extract(r"^\s*(\d+)")[0]
    s = pd.to_numeric(s, errors="coerce")
    s = s.where(s <= max_valid)
    return s.astype("Float64")

def short_label(text, n=60):
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"\s*-\s*(XYZ|kompetitor)\s*$", "", text, flags=re.I).strip()
    return text if len(text) <= n else text[: n - 1] + "…"

def detect_bank(question):
    return "Kompetitor" if "kompetitor" in str(question).lower() else "XYZ"

def normalize_friend(meta):
    # Same normalization buat_metadata's read_friend_metadata applies, but on the
    # in-memory friend frame (so metadata.csv is a documented output, not an input).
    meta = meta.copy()
    meta.columns = [str(c).strip() for c in meta.columns]
    for col in ("variable","question","Dashboard Role","Touchpoint","Construct"):
        if col not in meta.columns:
            meta[col] = ""
        meta[col] = meta[col].astype(str).str.strip()
    return meta

def read_question_row(path):
    # Fallback question text from the raw header rows (for vars absent from metadata).
    try:
        raw = pd.read_csv(path, sep=";", header=None, nrows=2,
                          encoding="utf-8-sig", dtype=str)
        if len(raw) >= 2:
            quests = raw.iloc[0].fillna("").tolist()
            codes = raw.iloc[1].fillna("").astype(str).str.strip().tolist()
            return {c: q for c, q in zip(codes, quests) if c}
    except Exception:
        pass
    return {}

In [12]:
def build_metadata(data_path, meta_df):
    data = pd.read_csv(data_path, sep=";", header=1, low_memory=False,
                       encoding="utf-8-sig")
    data.columns = [str(c).strip() for c in data.columns]
    meta = normalize_friend(meta_df)
    q_fallback = read_question_row(data_path)
    data_cols = set(data.columns)
    rows, seen = [], set()

    def n_terisi(var, scale):
        if var not in data_cols:
            return 0
        return int(to_score(data[var], scale).notna().sum())

    def add_row(var, question, section, touchpoint, subgroup, role, scale, include):
        var = str(var).strip()
        if not var or var in seen:
            return
        seen.add(var)
        ada = var in data_cols
        n = n_terisi(var, scale) if ada else 0
        rows.append({
            "variable": var,
            "question": re.sub(r"\s+", " ", str(question)).strip(),
            "label": short_label(question),
            "bank": detect_bank(question),
            "section": section, "touchpoint": touchpoint, "subgroup": subgroup,
            "role": role, "scale_max": scale,
            "include": int(include and ada and n > 0),
            "n_terisi": n, "ada_di_data": "Ya" if ada else "TIDAK DITEMUKAN",
        })

    # 4a. KPI Ringkasan
    for var, label, bank, scale in KPI_VARS:
        q = ""
        hit = meta.loc[meta["variable"] == var, "question"]
        if len(hit):
            q = hit.iloc[0]
        q = q or q_fallback.get(var, "") or f"{label} - {bank}"
        add_row(var, q, "Ringkasan", "", "KPI Utama", "KPI", scale, True)

    # 4b. Overall Touchpoint Satisfaction
    ov = meta[meta["Construct"].str.lower() == "overall touchpoint satisfaction"]
    for _, r in ov.iterrows():
        tp = r["Touchpoint"]
        section = SECTION_OF_TP.get(tp, "Service Experience")
        add_row(r["variable"], r["question"], section, tp, "Overall", "Overall",
                6, detect_bank(r["question"]) == "XYZ")

    # 4c. Attributes (drilldown) per touchpoint
    def attrs_of(tp):
        sub = meta[(meta["Dashboard Role"].str.lower() == "drilldown")
                   & (meta["Touchpoint"] == tp)]
        if sub.empty and tp == "Brand Image":
            # Brand Image has no Drilldown role in the friend layer -> use the T_C1B prefix
            sub = meta[(meta["Touchpoint"] == "Brand Image")
                       & (meta["variable"].str.startswith("T_C1B"))]
        return sub

    for tp in TP_ORDER:
        section = SECTION_OF_TP[tp]
        for _, r in attrs_of(tp).iterrows():
            add_row(r["variable"], r["question"], section, tp,
                    classify_subgroup(tp, r["question"]), "Atribut",
                    6, detect_bank(r["question"]) == "XYZ")

    # 4d. Filters & profile
    for var, label, kelompok in FILTER_VARS:
        q = ""
        hit = meta.loc[meta["variable"] == var, "question"]
        if len(hit):
            q = hit.iloc[0]
        add_row(var, q or label, "Filter & Profil", "", kelompok, "Filter", 0, True)
    for var in PROFIL_VARS:
        for r in rows:
            if r["variable"] == var:
                r["role"] = "Filter+Profil"


    # 4e. Loyalty Drivers (T_H* XYZ items only)
    #     Collected but previously orphaned: 15 XYZ driver dimensions that explain
    #     *why* customers are loyal, not just whether they will stay (F1A).
    ld = meta[
        meta["variable"].str.startswith("T_H") &
        ~meta["question"].str.lower().str.contains("kompetitor")
    ]
    for _, r in ld.iterrows():
        add_row(r["variable"], r["question"], "Ringkasan", "",
                "Loyalty Driver", "Loyalty Driver", 6, True)

    # 4f. Brand Image Importance (T_C1A* — importance ratings, scale 1-6 SANGAT PENTING)
    #     Enables Importance-Performance Analysis alongside T_C1B performance scores.
    #     Previously tagged Construct='Other' and never collected.
    bi_imp = meta[meta["variable"].str.startswith("T_C1A")]
    for _, r in bi_imp.iterrows():
        add_row(r["variable"], r["question"], "Brand Image", "Brand Image",
                classify_subgroup("Brand Image", r["question"]), "Importance", 6, True)

    out = pd.DataFrame(rows)
    # tidy sort: section -> touchpoint -> subgroup -> bank -> label
    out["__s"] = out["section"].map({s: i for i, s in enumerate(SECTION_ORDER)})
    out["__t"] = out["touchpoint"].map({t: i for i, t in enumerate(TP_ORDER)}).fillna(-1)
    out = (out.sort_values(["__s", "__t", "subgroup", "bank", "label"])
              .drop(columns=["__s", "__t"]).reset_index(drop=True))
    return out

dash = build_metadata(DATA, metadata)
dash.to_csv(OUT_DIR / "metadata_dashboard.csv", index=False,
            encoding="utf-8-sig", lineterminator="\r\n")   # CRLF + BOM, matches committed
print(f"wrote {OUT_DIR/'metadata_dashboard.csv'}  ({len(dash)} rows)")
dash.head()

wrote c:\Users\Muhammad Hafiz F\Documents\ali\deka done\metadata\metadata_dashboard.csv  (479 rows)


,variable,question,label,bank,section,touchpoint,subgroup,role,scale_max,include,n_terisi,ada_di_data
0,F1B,Ke depannya saya pasti akan tetap menggunakan ...,Ke depannya saya pasti akan tetap menggunakan ...,Kompetitor,Ringkasan,,KPI Utama,KPI,6,1,546,Ya
1,G1C,NPS bank kompetitor,NPS bank kompetitor,Kompetitor,Ringkasan,,KPI Utama,KPI,10,1,546,Ya
2,E1B,Tingkat Kepuasan terhadap Bank kompetitor,Tingkat Kepuasan terhadap Bank kompetitor,Kompetitor,Ringkasan,,KPI Utama,KPI,6,1,546,Ya
3,F1A,Ke depannya saya pasti akan tetap menggunakan ...,Ke depannya saya pasti akan tetap menggunakan ...,XYZ,Ringkasan,,KPI Utama,KPI,6,1,1730,Ya
4,G1A,NPS Bank XYZ,NPS Bank XYZ,XYZ,Ringkasan,,KPI Utama,KPI,10,1,1730,Ya


### 4.1 · The editable Excel control surface

`write_excel` mirrors `buat_metadata.py`: a styled **Metadata** sheet (frozen header,
per-section shading, an `include` 0/1 dropdown, auto-filter), a formula-driven **Ringkasan**
summary, and a **Petunjuk** (instructions) sheet. The app prefers the `.xlsx` over the `.csv`
when both exist, so editing `include`/`subgroup`/`label` here re-shapes the dashboard without
touching code. (The `.xlsx` bytes are not reproducible — openpyxl embeds timestamps — but the
Metadata-sheet *values* are identical to the committed file.)

In [13]:
def write_excel(out, path):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation

    C_DARK, C_PALE = "0F4C81", "D6E9F8"
    thin = Side(style="thin", color="B7CCE3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    f_head = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    f_body = Font(name="Arial", size=10)

    wb = Workbook()
    ws = wb.active
    ws.title = "Metadata"
    cols = list(out.columns)
    ws.append(cols)
    for _, r in out.iterrows():
        ws.append([r[c] for c in cols])
    for j, c in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=j)
        cell.font = f_head
        cell.fill = PatternFill("solid", start_color=C_DARK)
        cell.alignment = Alignment(vertical="center")
    widths = {"variable": 14, "question": 64, "label": 46, "bank": 11,
              "section": 18, "touchpoint": 18, "subgroup": 24, "role": 13,
              "scale_max": 10, "include": 9, "n_terisi": 9, "ada_di_data": 16}
    for j, c in enumerate(cols, start=1):
        ws.column_dimensions[get_column_letter(j)].width = widths.get(c, 14)
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=len(cols)):
        for cell in row:
            cell.font = f_body
            cell.border = border
            if cols[cell.column - 1] in ("question", "label"):
                cell.alignment = Alignment(wrap_text=True, vertical="top")
    sec_col = cols.index("section") + 1
    prev, shade = None, False
    for i in range(2, ws.max_row + 1):
        cur = ws.cell(row=i, column=sec_col).value
        if cur != prev:
            shade, prev = not shade, cur
        if shade:
            for j in range(1, len(cols) + 1):
                ws.cell(row=i, column=j).fill = PatternFill("solid", start_color="EFF6FD")
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(cols))}{ws.max_row}"
    inc_col = get_column_letter(cols.index("include") + 1)
    dv = DataValidation(type="list", formula1='"0,1"', allow_blank=False)
    ws.add_data_validation(dv)
    dv.add(f"{inc_col}2:{inc_col}{ws.max_row}")

    ws2 = wb.create_sheet("Ringkasan")
    ws2.append(["Section", "Jumlah Atribut (include=1)", "Jumlah Overall", "Total Variabel"])
    for j in range(1, 5):
        c = ws2.cell(row=1, column=j)
        c.font = f_head
        c.fill = PatternFill("solid", start_color=C_DARK)
    last = ws.max_row
    sections = [s for s in SECTION_ORDER if s in set(out["section"])]
    for i, s in enumerate(sections, start=2):
        ws2.cell(row=i, column=1, value=s).font = f_body
        ws2.cell(row=i, column=2).value = (
            f'=COUNTIFS(Metadata!$E$2:$E${last},$A{i},'
            f'Metadata!$H$2:$H${last},"Atribut",Metadata!$J$2:$J${last},1)')
        ws2.cell(row=i, column=3).value = (
            f'=COUNTIFS(Metadata!$E$2:$E${last},$A{i},'
            f'Metadata!$H$2:$H${last},"Overall",Metadata!$J$2:$J${last},1)')
        ws2.cell(row=i, column=4).value = f'=COUNTIF(Metadata!$E$2:$E${last},$A{i})'
        for j in range(2, 5):
            ws2.cell(row=i, column=j).font = f_body
    for j, w in enumerate([22, 26, 16, 14], start=1):
        ws2.column_dimensions[get_column_letter(j)].width = w
    ws2.freeze_panes = "A2"

    ws3 = wb.create_sheet("Petunjuk")
    tips = [
        "CARA MEMAKAI FILE INI", "",
        "1. Kolom 'include' = 1 berarti variabel DITAMPILKAN di dashboard, 0 berarti disembunyikan.",
        "   Ubah ke 0 untuk atribut yang tidak ingin dimunculkan agar tampilan tetap ringan.",
        "2. Kolom 'subgroup' adalah sub-kategori (mis. Toilet, Parkir, Ruang Tunggu).",
        "   Hasil pengelompokan otomatis — silakan rapikan/ganti namanya bila kurang pas.",
        "3. Kolom 'label' adalah teks singkat yang muncul di grafik. Boleh diedit.",
        "4. Jangan mengubah kolom 'variable' (harus sama persis dengan kode kolom di file data).",
        "5. Baris bank = 'Kompetitor' untuk atribut sengaja include=0; dashboard hanya memakai",
        "   E1B, F1B, G1C (KPI) untuk grafik perbandingan XYZ vs Kompetitor.",
        "6. 'ada_di_data' = TIDAK DITEMUKAN artinya kode variabel tidak ada di file data —",
        "   periksa kembali penulisan kodenya.",
        "7. Setelah selesai mengedit, cukup SIMPAN file ini (Ctrl+S).",
        "   Dashboard membaca 'metadata_dashboard.xlsx' lebih dulu; bila tidak ada,",
        "   baru membaca 'metadata_dashboard.csv'.",
    ]
    for i, t in enumerate(tips, start=1):
        c = ws3.cell(row=i, column=1, value=t)
        c.font = Font(name="Arial", size=11, bold=(i == 1),
                      color=C_DARK if i == 1 else "000000")
    ws3.column_dimensions["A"].width = 110
    ws3.sheet_view.showGridLines = False
    wb.save(path)

write_excel(dash, OUT_DIR / "metadata_dashboard.xlsx")
print(f"wrote {OUT_DIR/'metadata_dashboard.xlsx'}")

wrote c:\Users\Muhammad Hafiz F\Documents\ali\deka done\metadata\metadata_dashboard.xlsx


## 5 · Self-verification

The notebook documents its own correctness: it recomputes the dashboard's headline numbers
straight from the raw data and asserts them, then prints the attribute count per sub-category
(the same summary the old `buat_metadata.py` printed).

Expected headline (unfiltered Ringkasan): **CSAT 5.89 / 6 · NPS 81 · 1,730 responden**.

In [14]:
# Headline KPIs, computed exactly as dashboard.py does (to_score / nps).
csat = to_score(df["E1A"], 6).dropna().mean()
g = to_score(df["G1A"], 10).dropna()
nps = (g >= 9).mean() * 100 - (g <= 6).mean() * 100
n_resp = len(df)

print(f"CSAT       : {csat:.2f} / 6")
print(f"NPS        : {nps:.0f}")
print(f"Responden  : {n_resp:,}")

assert f"{csat:.2f}" == "5.89", f"CSAT changed: {csat:.2f}"
assert f"{nps:.0f}" == "81",   f"NPS changed: {nps:.0f}"
assert n_resp == 1730,          f"Responden changed: {n_resp}"

assert len(metadata) == 632, f"friend rows changed: {len(metadata)}"
assert len(dash) == 518,     f"dashboard rows changed: {len(dash)}"
print("\nAll headline assertions passed.")

CSAT       : 5.89 / 6
NPS        : 81
Responden  : 1,730

All headline assertions passed.


In [15]:
# Attributes per sub-category (n = total rows, tampil = include=1).
summary = (dash[dash["role"] == "Atribut"]
           .groupby(["section", "touchpoint", "subgroup"])
           .agg(n=("variable", "size"), tampil=("include", "sum")))
print(summary.to_string())
missing = dash[dash["ada_di_data"] != "Ya"]
print("\nVariabel tidak ditemukan di data:",
      "tidak ada" if missing.empty else f"\n{missing[['variable','section']].to_string(index=False)}")

                                                                   n  tampil
section            touchpoint          subgroup                             
ATM Experience     ATM                 ATM Umum                   12       8
                                       Fitur & Transaksi          12       8
                                       Keamanan                    9       6
                                       Keandalan Mesin            12       8
                                       Kenyamanan & Kebersihan     3       2
                                       Ketersediaan & Lokasi       6       4
Branch Facilities  Branch Facilities   Banking Hall & Kenyamanan  15      10
                                       Fasilitas Lain             18      12
                                       Parkir                     12       8
                                       Ruang Tunggu & Antrian     24      16
                                       Sarana Pendukung            3       2